# 03 - Context Engineering: measuring what survives, not what shrinks

Notebooks 01 and 02 fixed how the harness *learns*. This notebook fixes how it
*forgets* - and, more importantly, how you find out what it forgot.

The design review's complaint about the original compaction and offloading
sections was not that they were wrong, but that they measured the wrong thing:

| Review gap (8.1 / 8.2b) | This notebook |
|---|---|
| Card judged by character count | **Fidelity probe**: is the fact from turn 4 still there at turn 40, when it is finally needed? |
| "The bytes are recoverable" declared a win | **Agent in the loop**: the model must notice the reference, fetch it, and use the payload |
| Offloading and promotion never tested together | Offloaded blobs are **barred from promotion**, and the notebook proves it |

A smaller card is trivially easy to produce - throw everything away and the chart
looks fantastic. The only interesting question is what it still knows.

## 0 · Setup

Connect, verify notebook 01's artifacts, and check in on notebook 02 without
insisting on it - this notebook needs 02's scratch table, but not its scheduler.

In [ ]:
import json
import random

import matplotlib.pyplot as plt

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import context, promote

settings = load_settings()
conn = get_connection(settings)
require(conn, "01_self_improving_copilot")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
# Langfuse is optional all the way through: every invoke passes config=CFG.
CFG = {"callbacks": [HANDLER]} if HANDLER else {}
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

Notebook 02's status is reported, not enforced. Its scheduler jobs need
`CREATE JOB`, which is a database grant, not a context-engineering concept -
failing this notebook over it would be the wrong dependency. The one thing we
genuinely need from 02 is its scratch table, so we backfill that if it is absent.

In [ ]:
for desc, passed in verify(conn, "02_scheduled_briefings"):
    print(f"  {'✓' if passed else '·'} {desc}")


def ddl(sql):
    """Idempotent DDL: ORA-00955 (name already used) is the success case."""
    with conn.cursor() as cur:
        try:
            cur.execute(sql)
        except Exception as exc:
            if "ORA-00955" not in str(exc):
                raise


ddl("""CREATE TABLE HARNESS_SCRATCH (
         PATH        VARCHAR2(400) PRIMARY KEY,
         CONTENT     CLOB,
         STATUS      VARCHAR2(1) DEFAULT 'N' NOT NULL,
         CREATED_AT  TIMESTAMP DEFAULT SYSTIMESTAMP)""")
ok("notebook 02 scratch store available (backfilled if it was missing)")

## 1 · A long inspection season

Compaction only gets interesting when the transcript outgrows the window, so we
need length. Generating 40 turns with an LLM would be slow, expensive, and
different on every run - and none of that variation would teach anything. The
season below is built deterministically from *real* findings in
`CITY_INSPECTION_FINDING`, seeded so every learner compacts the same conversation.

Three turns carry a **planted fact**: a specific, checkable detail we will come
back for at the very end, long after compaction has had every opportunity to
discard it.

In [ ]:
ddl("""CREATE TABLE HARNESS_TRANSCRIPT (
         TURN_NO          NUMBER PRIMARY KEY,
         SPEAKER          VARCHAR2(20),
         CONTENT          CLOB,
         PLANTED_FACT_ID  VARCHAR2(10))""")

with conn.cursor() as cur:
    cur.execute("DELETE FROM HARNESS_TRANSCRIPT")
    cur.execute(
        "SELECT asset_id, category, severity, DBMS_LOB.SUBSTR(description, 400, 1)"
        "  FROM CITY_INSPECTION_FINDING ORDER BY finding_id"
    )
    FINDINGS = cur.fetchall()

check(len(FINDINGS) > 20, f"{len(FINDINGS)} real findings available to narrate")

In [ ]:
# The three facts the probe will hunt for, and the turns they land on.
PLANTED = {
    4:  ("f1", "Pier 3 bearing plate is corroded through; it is the single reason "
               "Harbor Bridge was prioritised for Q3."),
    11: ("f2", "The Riverside Culvert inspection used the 2019 rating scale, so its "
               "grades are not comparable with the rest of this season."),
    19: ("f3", "Inspector Okafor signed off every North District report this season, "
               "so a single reviewer bias applies to all of them."),
}

rng = random.Random(20260722)
TURNS = []
for turn_no in range(1, 41):
    if turn_no in PLANTED:
        fact_id, text = PLANTED[turn_no]
        TURNS.append((turn_no, "inspector", text, fact_id))
        continue
    asset, category, severity, desc = rng.choice(FINDINGS)
    TURNS.append((
        turn_no,
        "inspector" if turn_no % 2 else "coordinator",
        f"Turn {turn_no}: on {asset}, a {severity.lower()} {category.lower()} issue - "
        f"{(desc or '').strip()}",
        None,
    ))

with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO HARNESS_TRANSCRIPT (turn_no, speaker, content, planted_fact_id)"
        " VALUES (:1, :2, :3, :4)",
        TURNS,
    )
conn.commit()

check(len(TURNS) == 40, "40-turn season written")
check(sum(1 for t in TURNS if t[3]) == 3, "3 planted facts, at turns 4, 11 and 19")

## 2 · Where the card lives

Every compaction writes a **new version** rather than overwriting the last one.
That is what makes the size chart and the fidelity probe possible after the fact:
you can see not just the final card, but the moment a fact fell out of it.

In [ ]:
ddl("""CREATE TABLE HARNESS_CARD (
         CARD_VERSION      NUMBER PRIMARY KEY,
         CREATED_AT        TIMESTAMP DEFAULT SYSTIMESTAMP,
         TURNS_COVERED     NUMBER,
         CARD_JSON         CLOB,
         TRANSCRIPT_CHARS  NUMBER,
         CARD_CHARS        NUMBER)""")

with conn.cursor() as cur:
    cur.execute("DELETE FROM HARNESS_CARD")
conn.commit()


def save_card(version, turns_covered, card, transcript_chars):
    """Persist one compaction version. Returns the rendered card text."""
    body = context.render_card(card)
    # ✏️ TODO(1): insert the card version, and return `body`.
    #   Columns: CARD_VERSION, TURNS_COVERED, CARD_JSON, TRANSCRIPT_CHARS, CARD_CHARS.
    #   CARD_CHARS is len(body) - the size half of the measurement.
    #   ... your code here ...
    return body


def latest_card():
    with conn.cursor() as cur:
        cur.execute(
            "SELECT card_json FROM HARNESS_CARD"
            " ORDER BY card_version DESC FETCH FIRST 1 ROWS ONLY"
        )
        row = cur.fetchone()
    return context.parse_card(row[0].read() if hasattr(row[0], "read") else row[0]) if row else None


EMPTY_CARD = {"facts": [], "decisions": [], "open_questions": []}
ok("card store ready (versioned, never overwritten)")

## 3 · Compaction

We walk the season one turn at a time. Whenever the pending transcript reaches the
budget, the model folds it into the running card and the buffer resets.

Two design choices matter more than the prompt:

1. **The card has a fixed schema** (`facts`, `decisions`, `open_questions`). A
   free-form summary cannot be probed; a structured one can be, by set membership.
2. **The merge is additive** (`context.merge_card`). The model proposes; it does
   not get to silently delete. A fact leaves the card only when the model
   explicitly corrects it - which means anything that *does* go missing is a real
   finding about compaction, not an artefact of us overwriting the card each time.

In [ ]:
COMPACTION_PROMPT = (
    "You are maintaining a compaction card for a bridge-inspection season.\n"
    "Return ONLY JSON with keys: facts, decisions, open_questions.\n"
    "  facts: list of {{id, text, turn}} - durable, specific claims. Reuse an\n"
    "    existing id to correct it; invent a new id (f4, f5, ...) otherwise.\n"
    "  decisions: list of strings. open_questions: list of strings.\n"
    "Keep anything a colleague would need in three months.\n\n"
    "CURRENT CARD:\n{card}\n\nNEW TURNS:\n{turns}"
)


def compact(card, pending_turns):
    """Fold pending turns into the card. Returns the merged card."""
    prompt = COMPACTION_PROMPT.format(
        card=context.render_card(card),
        turns="\n".join(f"[{n}] {who}: {text}" for n, who, text, _ in pending_turns),
    )
    # ✏️ TODO(2): call the model, parse its card, and merge it into `card`.
    #   Use CHAT.invoke(prompt, config=CFG), context.parse_card, context.merge_card.
    #   Strip any ```json fence before parsing.
    #   ... your code here ...
    return card

ok("compaction step defined")

In [ ]:
card = dict(EMPTY_CARD)
pending, transcript_chars, version = [], 0, 0

for turn in TURNS:
    pending.append(turn)
    transcript_chars += len(turn[2])
    if context.compaction_due(sum(len(t[2]) for t in pending)):
        version += 1
        card = compact(card, pending)
        save_card(version, turn[0], card, transcript_chars)
        print(f"  v{version}: compacted through turn {turn[0]} - "
              f"{len(card['facts'])} facts, {len(context.render_card(card))} chars")
        pending = []

if pending:  # never strand the tail of the season
    version += 1
    card = compact(card, pending)
    save_card(version, TURNS[-1][0], card, transcript_chars)
    print(f"  v{version}: final fold through turn {TURNS[-1][0]}")

check(version >= 2, f"{version} card versions written across the season")

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT card_version, transcript_chars, card_chars"
        "  FROM HARNESS_CARD ORDER BY card_version"
    )
    rows = cur.fetchall()

versions = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(versions, [r[1] for r in rows], marker="o", label="transcript (cumulative)")
ax.plot(versions, [r[2] for r in rows], marker="o", label="compaction card")
ax.set_yscale("log")
ax.set_xlabel("card version")
ax.set_ylabel("characters (log scale)")
ax.set_title("What compaction saves")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

ratio = rows[-1][1] / max(rows[-1][2], 1)
ok(f"final card is {ratio:.1f}x smaller than the transcript it stands for")

### Read that chart sceptically

It is a real result and it is **not sufficient**. Every point on it would look
*better* if compaction simply threw more away - the best possible curve belongs to
a card that kept nothing at all. Size tells you what compaction cost you to store.
It says nothing about what it cost you to know.

So we go and ask.

## 4 · The probe: is it still in there?

Turn 4 planted a reason. Thirty-six turns later, we ask the question whose answer
*requires* that reason - with nothing to answer from but the card.

Two levels of scoring, because they can disagree in an interesting way:

- **Structural**: `context.card_fidelity` asks whether the fact id survived at all.
- **Functional**: does the model's answer actually contain the planted detail?

A fact can survive structurally and still be useless if compaction blurred it into
something too generic to answer with.

In [ ]:
PROBES = {
    "f1": ("Which pier did we flag, and what single reason put Harbor Bridge"
           " at the top of the Q3 list?", "pier 3"),
    "f2": ("Are all of this season's grades comparable with each other? If not,"
           " which inspection is the exception and why?", "2019"),
    "f3": ("Whose sign-off should a reviewer be careful about, and across which"
           " district?", "okafor"),
}

final_card = latest_card()
hits, misses, recall = context.card_fidelity(list(PROBES), final_card)
print(f"structural recall: {recall:.0%}  (kept: {hits or '-'}, lost: {misses or '-'})")

answered = []
# ✏️ TODO(3): for each probe, ask the model using ONLY the card as context,
#   and record whether the expected detail appears in its answer.
#   ... your code here ...

functional = sum(answered) / len(PROBES)
print(f"functional recall: {functional:.0%}")

# Deliberately not 100%: compaction really does lose things, and a notebook that
# only passes when nothing is ever lost would be teaching a fiction.
check(recall >= 2 / 3, f"card retained {recall:.0%} of facts planted up to 36 turns earlier")
if misses:
    print(f"lost along the way: {', '.join(misses)} - look at which card version dropped it")

### What the misses tell you

Facts survive compaction when the card has a **slot** for them - that is the whole
reason the card has a fixed schema rather than being a paragraph of prose. The
first thing lossy summarisation discards is the category with the weakest slot,
which is almost always `open_questions`: they read as incidental, and nothing in
the transcript ever refers back to them.

If a fact went missing above, find the version where it vanished
(`SELECT card_version, card_json FROM HARNESS_CARD ORDER BY card_version`) and look
at what the model was folding in at that moment. That, not the character count, is
the feedback loop that improves a compaction prompt.